# Lekcija 05 - Agentični RAG


## Nastavitev

Ta zvezek prikazuje vzorec Agentic RAG (povečana generacija z iskanjem) z uporabo Microsoft Agent Framework.

**Predpogoji:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — vaš konec storitve Azure AI Search
- `AZURE_SEARCH_API_KEY` — vaš API ključ za Azure AI Search
- Azure OpenAI nameščena preko okoljskih spremenljivk
- Azure CLI prijavljen (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kaj je Agentic RAG?

Tradicionalni RAG sledi fiksni poti: najprej pridobi dokumente, nato generira odgovor. **Agentic RAG** gre korak dlje, saj agentu daje avtonomijo, da odloči, **kdaj** in **kako** pridobiti informacije.

Z Agentic RAG lahko agent:
- **Odloči**, ali je pridobivanje informacij potrebno pred odgovarjanjem na vprašanje
- **Izbere**, kateri vir podatkov ali orodje bo povprašal
- **Ocenjuje** pridobljene rezultate in izvede dodatna pridobivanja, če prvi poskus ni zadosten
- **Združi** informacije iz več korakov pridobivanja v skladen odgovor

To naredi agenta bolj prilagodljivega in natančnega v primerjavi s statičnim postopkom pridobi-potem-generiraj.


## Ustvarjanje orodja za iskanje

V Agentic RAG so zunanje podatkovne vire ovite kot **orodja**, ki jih agent lahko po potrebi pokliče. To agentu omogoča, da obravnava pridobivanje podatkov kot še eno dejanje, ki ga lahko opravi, namesto kot obvezni korak.

Spodaj definiramo bazo znanja o potovanjih in jo predstavimo kot orodje, ki ga lahko agent pokliče za iskanje informacij o destinacijah.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Izdelava RAG agenta

Zdaj ustvarimo agenta, ki je navodilo, da **vedno pridobi informacije, preden odgovori**. Agent uporablja orodje `search_travel_knowledge`, da temelji svoje odgovore na zbirki znanja namesto, da bi se zanašal na svoje lastne podatke usposabljanja.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativno iskanje — vzorec ustvarjalca in preverjevalca

Ključna prednost Agentic RAG je **iterativno iskanje**. Agent lahko izvede več krogov iskanja, da preveri, izboljša ali razširi svoje začetne ugotovitve — podobno kot delovni proces »ustvarjalec-preverjevalec«:

1. **Korak ustvarjalca**: Agent pridobi začetne informacije in pripravi osnutek odgovora.
2. **Korak preverjevalca**: Agent opravi dodatna iskanja, da preveri podrobnosti ali zapolni vrzeli.

Spodaj je agentu zastavljeno vprašanje, ki zahteva primerjavo več destinacij, zaradi česar je spodbujen k večkratnemu iskanju.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Povzetek

V tej lekciji ste se naučili, kako zgraditi sistem **Agentic RAG** z uporabo Microsoft Agent Framework:

- **Agentic RAG** omogoča agentom, da avtonomno odločajo, kdaj pridobiti informacije, zaradi česar je pridobivanje dinamično in ne fiksno.
- **Orodja kot podatkovni viri**: Zunanji baze znanja (kot je Azure AI Search) so zavite kot orodja, ki jih agent lahko pokliče.
- **Iterativno pridobivanje**: Vzorec maker-checker omogoča agentu, da izvede več krogov pridobivanja — iskanje, preverjanje in izboljševanje — preden ustvari končni odgovor.

V produkciji bi namesto podatkovne baze `TRAVEL_KNOWLEDGE_BASE` v pomnilniku uporabili pravi indeks Azure AI Search za obdelavo obsežnega iskanja po potovalnih dokumentih.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o omejitvi odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, prosimo, upoštevajte, da avtomatski prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku velja za vsebinsko zavezujoč vir. Za pomembne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za morebitne nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
